# First Try of Plotly

In this notebook, plotly will be exploited to perform 3D plots. 
Compton events are used for demonstration.

In [1]:
from podio.root_io import Reader

compton_event_path = "./compton_digi.root"
reader = Reader(compton_event_path)

events = reader.get("events")

Module libc not found.


Welcome to JupyROOT 6.28/06


/opt/podio/v00-17-02/python/podio/EventStore.py:4: FutureWarning: The EventStore based I/O model is deprecated and will be removed. Switch to the Frame based model.
  warnings.warn("The EventStore based I/O model is deprecated and will be removed. Switch to the Frame based model.",


We first deal with the first event.

In [2]:
import numpy as np
from RecoUtils import decodeRawHits, vec3d2mat
from Hit3DBuilder import buildHitPairs

event = events[0]
traj_points = event.get("TpcSimHits")
plain_hits = event.get("TpcHits")
wave_hits = event.get("TpcWaveformHits")

In [3]:
traj_coor = vec3d2mat(traj_points.position())
traj_edep = np.array(traj_points.EDep())

decoded_hits = decodeRawHits(plain_hits)
rebuilt_hits = buildHitPairs(decoded_hits.x_hits, decoded_hits.y_hits)

Info in <TGeoManager>: Changing system of units to Geant4 units (mm, ns, MeV).


## Plotly

Use plotly to produce 3D plots.

Just like plots in ComptonDemo.ipynb, we put trajactory points, 

In [4]:
import plotly.express as px
import plotly.graph_objects as go

In [5]:
import pandas as pd

drift_velocity = 60

rebuilt_hits["z_pos"] = rebuilt_hits.apply(
    lambda df: 0.5
    * ((df.x_z + df.y_z) - (df.x_time + df.y_time) / 1000 * drift_velocity),
    axis=1,
)
rebuilt_hits["eDep"] = rebuilt_hits["x_edep"] + rebuilt_hits["y_edep"]
df_rebuilt = rebuilt_hits[["x_pos", "y_pos", "z_pos", "eDep"]]
df_rebuilt.columns = ["x", "y", "z", "eDep"]
df_rebuilt.loc[:, "Type"] = "Rebuilt"

df_traj = pd.DataFrame(
    {
        "x": traj_coor[:, 0],
        "y": traj_coor[:, 1],
        "z": traj_coor[:, 2],
        "eDep": traj_edep,
        "Type": "Trajactory",
    }
)

df = pd.concat([df_rebuilt, df_traj])

/tmp/ipykernel_2328522/1798704004.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_rebuilt.loc[:, "Type"] = "Rebuilt"


In [21]:
fig = px.scatter_3d(
    df, x="x", y="y", z="z", color="eDep", symbol="Type", width=800, height=600
)
fig.update_layout(legend=dict(x=1.2, y=0.5))
fig.show()

In [7]:
fig2 = px.scatter_3d(df_traj, x="x", y="y", z="z", color="eDep", opacity=0.4)
fig2.add_scatter3d(x=df_rebuilt["x"], y=df_rebuilt["y"], z=df_rebuilt["z"])
fig2.show()

In [20]:
from matplotlib import legend


st = go.Scatter3d(
    x=df_traj["x"],
    y=df_traj["y"],
    z=df_traj["z"],
    mode="markers",
    marker=dict(
        size=2,
        color=df_traj["eDep"],
        colorscale="Rainbow",
        opacity=0.5,
        coloraxis="coloraxis",
    ),
    name="Trajactory Points",
)
sr = go.Scatter3d(
    x=df_rebuilt["x"],
    y=df_rebuilt["y"],
    z=df_rebuilt["z"],
    mode="markers",
    marker=dict(
        size=3,
        color=df_traj["eDep"],
        colorscale="Rainbow",
        opacity=0.8,
        coloraxis="coloraxis",
    ),
    name="Rebuilt Points",
)
fig = go.Figure()
fig.update_layout(width=900, height=600, legend=dict(x=1.2, y=0.5))
fig.add_traces([st, sr])
fig.show()